# 🩸 Blood Cell Detector — YOLOv8n մարզում (BCCD)

Այս notebook-ը մարզում է YOLOv8n մոդելը՝ արյան քսուքի նկարներում **RBC**, **WBC** և **Platelets** բջիջները հայտնաբերելու համար։

Ձեզ պետք է միայն `BCCD_Dataset-master.zip` ֆայլը (https://github.com/Shenggan/BCCD_Dataset)։

**Սկսելուց առաջ.** մենյուից ընտրեք `Runtime → Change runtime type → T4 GPU → Save`։

Հետո բջիջները գործարկեք հերթով՝ վերևից ներքև (յուրաքանչյուր բջջի ձախ կողմի ▶ կոճակով)։

## Քայլ 1. Տեղադրում և GPU-ի ստուգում

Տեղադրում է Ultralytics-ը (նույն տարբերակը, ինչ ձեր համակարգչում է) և ստուգում՝ կա՞ արդյոք CUDA GPU։

In [ ]:
!pip install -q ultralytics==8.4.215

import torch
import ultralytics

print("ultralytics:", ultralytics.__version__)
print("torch:", torch.__version__)

if torch.cuda.is_available():
    device = 0
    print("GPU կա ✅:", torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("GPU չկա ⚠️ — մարզումը կաշխատի CPU-ով և շատ ավելի դանդաղ կլինի։")
    print("Միացրեք՝ Runtime → Change runtime type → T4 GPU, հետո նորից գործարկեք այս բջիջը։")

## Քայլ 2. ZIP ֆայլի վերբեռնում

Գործարկեք բջիջը, սեղմեք **Choose Files** և ընտրեք `BCCD_Dataset-master.zip` ֆայլը ձեր համակարգչից։ Ֆայլը կբացվի ինքնաշխատ։

In [ ]:
import glob
import os
import shutil
import zipfile

ZIP_PATH = "/content/BCCD_Dataset-master.zip"
RAW_DIR = "/content/bccd_raw"

if not os.path.exists(ZIP_PATH):
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("Ֆայլ չի վերբեռնվել։ Նորից գործարկեք այս բջիջը։")
    shutil.move(next(iter(uploaded)), ZIP_PATH)

if not zipfile.is_zipfile(ZIP_PATH):
    os.remove(ZIP_PATH)
    raise SystemExit("Սա .zip ֆայլ չէ։ Նորից գործարկեք այս բջիջը և ընտրեք BCCD_Dataset-master.zip-ը։")

shutil.rmtree(RAW_DIR, ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(RAW_DIR)

xml_files = sorted(glob.glob(f"{RAW_DIR}/**/Annotations/*.xml", recursive=True))
image_files = sorted(
    p for p in glob.glob(f"{RAW_DIR}/**/JPEGImages/*", recursive=True)
    if p.lower().endswith((".jpg", ".jpeg", ".png"))
)
print("XML annotation-ներ՝", len(xml_files))
print("Նկարներ՝", len(image_files))
if not xml_files or not image_files:
    raise SystemExit("Annotations կամ JPEGImages թղթապանակը չի գտնվել։ Ստուգեք, որ ճիշտ ZIP ֆայլն եք վերբեռնել։")

## Քայլ 3. Փոխարկում YOLO ձևաչափի և բաժանում

Pascal VOC XML-ները փոխարկում է YOLO label-ների, նկարները բաժանում է train 70% / validation 20% / test 10% մասերի (ֆիքսված seed-ով), ստեղծում է `data.yaml`-ը և ստուգում, որ բոլոր երեք դասերը կան յուրաքանչյուր մասում։

In [ ]:
import random
import xml.etree.ElementTree as ET
from collections import Counter

from PIL import Image

CLASSES = ["RBC", "WBC", "Platelets"]
DATASET_DIR = "/content/bccd_yolo"
DATA_YAML = f"{DATASET_DIR}/data.yaml"
SEED = 42


def xml_to_yolo(xml_path, image_path):
    """Վերադարձնում է (YOLO տողեր, բաց թողնված box-երի քանակ)։"""
    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError:
        return [], 0
    try:
        w, h = int(root.findtext("size/width")), int(root.findtext("size/height"))
    except (TypeError, ValueError):
        w = h = 0
    if w <= 0 or h <= 0:  # չափը XML-ում չկա՝ կարդում ենք նկարից
        try:
            with Image.open(image_path) as im:
                w, h = im.size
        except Exception:
            return [], 0

    lines, skipped = [], 0
    for obj in root.iter("object"):
        name = (obj.findtext("name") or "").strip()
        try:
            x1, y1, x2, y2 = (float(obj.findtext(f"bndbox/{k}")) for k in ("xmin", "ymin", "xmax", "ymax"))
        except (TypeError, ValueError):
            skipped += 1
            continue
        x1, x2 = max(0.0, min(x1, w)), max(0.0, min(x2, w))
        y1, y2 = max(0.0, min(y1, h)), max(0.0, min(y2, h))
        if name not in CLASSES or x2 <= x1 or y2 <= y1:
            skipped += 1
            continue
        lines.append(
            f"{CLASSES.index(name)} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} "
            f"{(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}"
        )
    return lines, skipped


# 1. Նկար + XML զույգերի փոխարկում
images_by_name = {os.path.splitext(os.path.basename(p))[0]: p for p in image_files}
samples, skipped_images, skipped_boxes = [], 0, 0
for xml_path in xml_files:
    stem = os.path.splitext(os.path.basename(xml_path))[0]
    image_path = images_by_name.get(stem)
    lines, skipped = xml_to_yolo(xml_path, image_path) if image_path else ([], 0)
    skipped_boxes += skipped
    if lines:
        samples.append((stem, image_path, lines))
    else:
        skipped_images += 1
print(f"Օգտագործելի նկարներ՝ {len(samples)} | բաց թողնված նկարներ՝ {skipped_images} | բաց թողնված box-եր՝ {skipped_boxes}")

# 2. Բաժանում 70 / 20 / 10 (ֆիքսված seed)
random.Random(SEED).shuffle(samples)
n_train, n_val = int(len(samples) * 0.7), int(len(samples) * 0.2)
splits = {
    "train": samples[:n_train],
    "valid": samples[n_train:n_train + n_val],
    "test": samples[n_train + n_val:],
}

shutil.rmtree(DATASET_DIR, ignore_errors=True)
for split, items in splits.items():
    os.makedirs(f"{DATASET_DIR}/{split}/images")
    os.makedirs(f"{DATASET_DIR}/{split}/labels")
    for stem, image_path, lines in items:
        shutil.copy(image_path, f"{DATASET_DIR}/{split}/images/{os.path.basename(image_path)}")
        with open(f"{DATASET_DIR}/{split}/labels/{stem}.txt", "w") as f:
            f.write("\n".join(lines) + "\n")

# 3. data.yaml
with open(DATA_YAML, "w") as f:
    f.write(f"path: {DATASET_DIR}\n")
    f.write("train: train/images\n")
    f.write("val: valid/images\n")
    f.write("test: test/images\n")
    f.write(f"nc: {len(CLASSES)}\n")
    f.write("names:\n")
    for i, name in enumerate(CLASSES):
        f.write(f"  {i}: {name}\n")
print(open(DATA_YAML).read())

# 4. Ստուգում՝ կարդալով ստեղծված ֆայլերը
for split in splits:
    n_images = len(os.listdir(f"{DATASET_DIR}/{split}/images"))
    label_paths = glob.glob(f"{DATASET_DIR}/{split}/labels/*.txt")
    counts = Counter()
    for path in label_paths:
        for line in open(path).read().split("\n"):
            if line:
                counts[CLASSES[int(line.split()[0])]] += 1
    print(f"{split}: {n_images} նկար, {len(label_paths)} label | " + ", ".join(f"{c}: {counts[c]}" for c in CLASSES))
    if n_images == 0 or n_images != len(label_paths):
        raise SystemExit(f"{split}: նկարների և label-ների քանակը չի համընկնում։")
    missing = [c for c in CLASSES if counts[c] == 0]
    if missing:
        raise SystemExit(f"{split}: բացակայում են դասեր՝ {missing}")
print("Dataset-ը պատրաստ է ✅")

## Քայլ 4. Մարզում

Մարզում է նախապես մարզված `yolov8n.pt` մոդելը՝ 15 epoch, նկարի չափը 416, batch 8։ T4 GPU-ով սա տևում է մի քանի րոպե։

In [ ]:
from pathlib import Path
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=DATA_YAML,
    epochs=15,
    imgsz=416,
    batch=8,
    device=device,
    seed=SEED,
    project="/content/runs",
    name="bccd",
    exist_ok=True,
)

BEST = Path(results.save_dir) / "weights" / "best.pt"
print("Լավագույն կշիռները՝", BEST)

## Քայլ 5. Գնահատում test նկարների վրա

Ստուգում է մարզված մոդելը test մասի վրա (այս նկարները մոդելը մարզման ընթացքում չի տեսել) և տպում precision, recall և mAP50 ցուցանիշները։

In [ ]:
best_model = YOLO(BEST)
metrics = best_model.val(data=DATA_YAML, split="test", imgsz=416, batch=8, device=device)

print()
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")
print(f"mAP50:     {metrics.box.map50:.3f}")
print("Մոդելի դասերը՝", best_model.names)

## Քայլ 6. `best.pt`-ի ներբեռնում

Ներբեռնում է մարզված մոդելը ձեր համակարգիչ։ Հետո `best.pt` ֆայլը տեղափոխեք նախագծի `models/` թղթապանակ և գործարկեք `streamlit run app.py`։

In [ ]:
from google.colab import files

shutil.copy(BEST, "/content/best.pt")
files.download("/content/best.pt")